# Assemble .fcs files and metadata from the PB1 panel

## Imports and output

In [20]:
from datetime import date # date for uploads

import hisepy # HISE SDK

import polars as pl # data frame handling
import flowio # flow cytometry file handling

import re # regular expressions

from io import StringIO # string reading - required to fix a panel file issue
import os # file/path utilities
import shutil # file utilities
import tarfile # file bundling in .tar files

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

## Helper functions

Generate a unique ID for file deposit

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

Helpers for finding the most recent batch and file timestamp to ensure we have the most recent version of our files

In [4]:
def get_latest_batch(df):
    batches = list(df['file.batchID'].unique())
    batches.sort()
    return batches[-1]

In [5]:
def file_to_time(fn):
    file_time = re.sub('AIFI-(.+)Z/.+','\\1',fn)
    file_time = re.sub('UCSD-(.+)Z/.+','\\1',file_time)
    file_time = re.sub('\\..+','',file_time)
    return file_time

Helpers for file caching. By default, `hisepy.cache_files` will re-retrieve each file, even if we've cached it before.

We'll also do our file caching queries in chunks of 50 files.

In [6]:
def find_nested_file(path):
    if os.path.isdir(path):
        subpath = os.listdir(path)[0]
        path = f'{path}/{subpath}'
        return find_nested_file(path)
    else:
        return path

In [7]:
def cache_chunk_once(ids):    
    # get cached files
    in_root = '/home/workspace/input/1918706177/'
    projects = os.listdir(in_root)
    cached_ids = {}
    for project in projects:
        project_ids = os.listdir(f'{in_root}/{project}')
        for project_id in project_ids:
            cached_ids[project_id] = f'{in_root}/{project}/{project_id}'
    
    cached_paths = []
    not_cached = []
    for uuid in ids:
        if not uuid in cached_ids.keys():
            not_cached.append(uuid)
        else:
            cache_path = find_nested_file(cached_ids[uuid])
            cached_paths.append(cache_path)

    if len(not_cached) > 0:
        new_cache = hisepy.cache_files(not_cached)
        cached_paths = cached_paths + new_cache
    
    return cached_paths

## Read sample metadata

A few adjustments need to be made to use the metadata:
- Trim drawDate to drawYear
- Add subject.ageGroup
- Select relevant columns

In [8]:
meta_uuid = 'af25e3e7-25c1-4476-afb4-926bd201db8f'

In [9]:
meta_file = hisepy.cache_files([meta_uuid])[0]

2026-03-14 15:35:05,083 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 15:35:09,794 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=3.344s


In [10]:
meta = pl.read_csv(meta_file)

In [11]:
meta.shape

(868, 19)

### drawDate -> drawYear

In [12]:
meta = meta.with_columns(
    pl.col('sample.drawDate').str.replace('-.+','')
).rename({'sample.drawDate':'sample.drawYear'})

### Add age groups

In [13]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}

In [14]:
meta = meta.with_columns(
    pl.Series(
        name = 'subject.ageGroup',
        values = [age_groups[c] for c in meta['cohort.cohortGuid']]
    )
)

### Select columns

In [15]:
keep_meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'subject.cmv',
    'sample.visitName',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit'
]

In [16]:
meta = meta.select(keep_meta_cols)

In [17]:
meta.head()

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64
"""BR1""","""BR1001""","""KT00001""","""Female""",1987,32,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",32,0
"""BR1""","""BR1002""","""KT00002""","""Male""",1991,28,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",28,0
"""BR1""","""BR1003""","""KT00003""","""Female""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0
"""BR1""","""BR1004""","""KT00004""","""Male""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0
"""BR1""","""BR1005""","""KT00006""","""Female""",1992,27,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",27,0


## Get panel definition
We'll include the panel composition in the output files for convenience

In [21]:
panel_uuid = "28d563bf-4b35-405d-9136-56b9f1d77f24"
panel_csv = hisepy.cache_files([panel_uuid])[0]

2026-03-14 15:39:20,094 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 15:39:24,473 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=2.979s


For this panel, there's an extra comma that will cause import problems - the clone ID for the CD319 antibody includes a comma. This ID is listed by some vendors without a comma, so it may have been accidentally incorporated by spreadsheet software. We'll drop the comma here before loading.

In [27]:
with open(panel_csv, 'r', encoding='utf-8') as file:
    file_content = file.read()
file_content = re.sub('235,614', '235614', file_content)

In [28]:
panel_meta = pl.read_csv(StringIO(file_content))

## Locate FCS files in HISE

For each sample, we'll need to filter for the file from the latest batch and with the most recent timestamp.

We'll work through the sample kit IDs in chunks to help manage our queries to HISE for so many files.

In [29]:
meta_chunks = meta.iter_slices(n_rows = 50)

In [30]:
pb1_list = []
for chunk in meta_chunks:
    chunk_files = hisepy.get_file_descriptors(
        query_dict = {
            'fileType': ['FlowCytometry'],
            'sampleKitGuid': chunk['sample.sampleKitGuid'].to_list(),
            'panel': ['PB1']
        }
    )['descriptors']
    pb1_list.append(pl.DataFrame(chunk_files))

2026-03-14 15:44:26,229 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling get_file_descriptors
2026-03-14 15:44:26,230 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling lookup_queryable_fields
2026-03-14 15:44:29,210 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished lookup_queryable_fields, success=True, time_elapsed=1.397s
2026-03-14 15:44:29,212 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling lookup_queryable_fields
2026-03-14 15:44:31,599 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished lookup_queryable_fields, success=True, time_elapsed=1.095s
2026-03-14 15:44:43,417 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished get_file_descriptors, success=True, time_elapsed=15.540s
2026-03-14 15:44:43,841 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling get_file_descriptors
2026-03-14 15:44:43,842 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling lookup_queryable_fields
2026-03-14 1

We only need some of the columns that are provided by HISE to find the most recent version and batch for each sample kit.

Note that the file time stamp is embedded in the file.name - see the helper function above for parsing.

In [33]:
keep_cols = [
    'file.availability',
    'file.batchID',
    'file.id',
    'file.majorVersion',
    'file.name',
    'file.panel',
    'sample.sampleKitGuid',
]

The output of our chunk-based queries, above, is a list of data frames per chunk. We'll select columns and concatenate these chunks here.

`desc` is short for `descriptors`.

In [34]:
common_pb1_list = []
for df in pb1_list:
    common_pb1_list.append(df.select(keep_cols))

In [35]:
desc = pl.concat(common_pb1_list)
desc.shape

(1717, 7)

Now we'll do the filtering for each kit to get the latest batch and time stamp. We'll also keep track of any sample kits that are in the sample metadata but weren't found in our HISE file queries.

In [36]:
missing_kits = {}

In [37]:
kit_list = []
missing_list = []
for sample_kit in meta['sample.sampleKitGuid']:
    kit_files = desc.filter(pl.col('sample.sampleKitGuid') == sample_kit)
    if kit_files.shape[0] == 0:
        print(f'No files for {sample_kit}; Skipping.')
        missing_list.append(sample_kit)
        continue
    else:
        # Filter for latest batch
        latest_batch = get_latest_batch(kit_files)
        kit_files = kit_files.filter(pl.col('file.batchID') == latest_batch)
        # Filter for most recent file
        kit_files = kit_files.with_columns(
            pl.Series(
                name = 'file_time',
                values = [file_to_time(x) for x in kit_files['file.name']]
            ).str.to_datetime("%Y-%m-%dT%H:%M:%S")
        ).sort('file_time', descending = True).head(1)

        kit_list.append(kit_files)

The output for each kit is a list again, so we'll concatenate these for downstream use.

In [38]:
pb1_desc = pl.concat(kit_list)

Are there any missing kits?

In [39]:
len(missing_list)

0

In [40]:
missing_kits['PB1'] = meta.filter(pl.col('sample.sampleKitGuid').is_in(missing_list))

We should get the same number of rows in the descriptors data frame as in the original metadata file if all sample kits are accounted for.

In [41]:
meta.shape

(868, 14)

In [42]:
pb1_desc.shape

(868, 8)

## Cache and assemble files

Split the file descriptors to get file ids, and run queries in chunks of 50 files.

In [43]:
pb1_chunks = pb1_desc.iter_slices(n_rows = 50)

In [44]:
pb1_fail_list = []
pb1_files = []

for pb1_chunk in pb1_chunks:
    ids = pb1_chunk['file.id'].to_list()
    
    try:
        fn = cache_chunk_once(ids)
        pb1_files = pb1_files + fn
    except:
        pb1_fail_list.append(pb1_chunk)

2026-03-14 16:02:04,442 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 16:03:54,698 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=108.543s
2026-03-14 16:03:54,850 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 16:07:28,216 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=103.793s
2026-03-14 16:07:28,393 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 16:09:15,207 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=105.521s
2026-03-14 16:09:15,367 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling cache_files
2026-03-14 16:11:00,650 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished cache_files, success=True, time_elapsed=103.942s
2026-03-14 16:11:00,839 INFO [hisepy.logging:175] loggin

In the end, we should get the same number of files as we had in pb1_desc:

In [48]:
pb1_desc.shape

(868, 8)

In [47]:
len(pb1_files)

868

### Transfer and structure outputs

We'll make a hierarchical file structure that will enable all of the panels and file groups to be untarred into the same folder structure.

File bundles will contain all samples for each age + sex + cmv group, and will have a metadata file for each group. We also need to have a file defining the panel in each bundle so that it's included in any particular bundle.

An example for one panel is shown below:

```
sound-life_flow-cytometry/
  PB1_panel/
    older-adult_female_cmv-negative/
      <subject>_<sample_kit>_<visit_name>_unmixed.fcs
    older-adult_female_cmv-positive/
    older-adult_male_cmv-negative/
    older-adult_male_cmv_positive/
    young-adult_female_cmv-negative/
    young-adult_female_cmv-positive/
    young-adult_male_cmv-negative/
    young-adult_male_cmv_positive/
```

First, we'll add the sample metadata to our descriptors so we can split files into the groups above.

In [49]:
file_df = pb1_desc.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
)

In [50]:
file_df.shape

(868, 21)

### Move files into a stage for .tar

Here, we'll move the files from the cache into the path structure defined above.

In [51]:
in_root = '/home/workspace/input/1918706177/'
out_base = 'sound-life_flow-cytometry/PB1_panel/'

Associate each uuid with the file location in the cache

In [52]:
all_cached_files = {}
for project in os.listdir(in_root):
    for uuid in pb1_desc['file.id']:
        if os.path.isdir(f'{in_root}/{project}/{uuid}'):
            all_cached_files[uuid] = find_nested_file(f'{in_root}/{project}/{uuid}')

Add cache file names to the file descriptors

In [53]:
panel_files = []

for uuid in pb1_desc['file.id']:
    if uuid in all_cached_files.keys():
        panel_files.append(all_cached_files[uuid])
    else:
        panel_files.append(None)

file_df = file_df.with_columns(
    pl.Series(
        name = 'in_file',
        values = panel_files
    )
)

Build directory structure file names. We'll make 3 columns here:

`out_dir`: The subdirectory based on:  
- `subject.ageGroup`\_`subject.biologicalSex`_`subject.cmv`/

`out_file`: The full target file name, based on:  
- `out_dir`/`subject.subjectGuid`\_`sample.sampleKitGuid`_`sample.visitName`_unmixed.fcs

`file.name`: The file path relative to the location of the metadata files in the file structure. This is the file name we'll provide to users.

In [54]:
file_df = file_df.with_columns(
    pl.col('subject.ageGroup').str.replace(' ', '-'),
    pl.col('sample.visitName').str.replace_all(' ', '-')
).with_columns(
    pl.concat_str(
        pl.lit(out_base),
        pl.col('subject.ageGroup').str.to_lowercase(),
        pl.lit('_'),
        pl.col('subject.biologicalSex').str.to_lowercase(),
        pl.lit('_cmv-'),
        pl.col('subject.cmv').str.to_lowercase(),
        pl.lit('/')
    ).alias('out_dir')
).with_columns(
    pl.concat_str(
        pl.col('out_dir'),
        pl.col('subject.subjectGuid'),
        pl.lit('_'),
        pl.col('sample.sampleKitGuid'),
        pl.lit('_'),
        pl.col('sample.visitName').str.to_lowercase(),
        pl.lit('_unmixed.fcs')
    ).alias('out_file')
).with_columns(
    pl.col('out_file').str.replace('sound-life_flow-cytometry/', '').alias('file.name')
)

### Make staging directories and move files

Make the output subdirectories

In [55]:
for out_dir in file_df['out_dir'].unique().to_list():
    if out_dir is not None:
        if not os.path.isdir(out_dir):
            os.makedirs(out_dir)

Copy the files to the staging directories

In [56]:
for in_file, out_file in zip(file_df['in_file'], file_df['out_file']):
    if not out_file is None:
        if not os.path.isfile(out_file):
            shutil.copy(in_file, out_file)

Add the panel metadata file

In [61]:
fcs_panel_file = 'sound-life_flow-cytometry/PB1_panel_feature_metadata.csv'
panel_meta.write_csv(fcs_panel_file)

## Get file-specific stats from within the .fcs files

In [62]:
file_df = file_df.with_columns(
    pl.Series(
        name = 'flow.event_count',
        values = [flowio.FlowData(f).event_count for f in file_df['out_file']]
    )
)

We'll rename the columns specific to the flow run to prepend `flow.` instead of `file.`

In [ ]:
file_df = file_df.rename({
    'file.panel': 'flow.panel',
    'file.batchID': 'flow.batchID'
})

Select the file and sample metadata columns we want to retain for download.

We'll also revert the space to dash replacement we used to generate filenames.

In [64]:
file_df = file_df.select(
    'out_dir', 'file.name', 'file.id', 
    'flow.panel', 'flow.batchID', 'flow.event_count',
    'subject.ageGroup', 'subject.biologicalSex', 'subject.cmv', 
    'cohort.cohortGuid', 
    'subject.subjectGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.race', 'subject.ethnicity',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'sample.daysSinceFirstVisit'
).with_columns(
    pl.col('subject.ageGroup').str.replace('-', ' ')
)

In [65]:
file_df.head()

out_dir,file.name,file.id,flow.panel,flow.batchID,flow.event_count,subject.ageGroup,subject.biologicalSex,subject.cmv,cohort.cohortGuid,subject.subjectGuid,subject.birthYear,subject.ageAtFirstDraw,subject.race,subject.ethnicity,sample.sampleKitGuid,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,str,i64,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64
"""sound-life_flow-cytometry/PB1_…","""PB1_panel/young-adult_female_c…","""8ecf2afd-29fc-409b-b3b4-3e5e49…","""PB1""","""B151""",350880,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1001""",1987,32,"""Caucasian""","""Non-Hispanic origin""","""KT00001""","""Flu-Year-1-Day-0""","""2019""",32,0
"""sound-life_flow-cytometry/PB1_…","""PB1_panel/young-adult_male_cmv…","""f5d96952-4fd5-48e1-a0ee-aa7132…","""PB1""","""B151""",494408,"""Young Adult""","""Male""","""Negative""","""BR1""","""BR1002""",1991,28,"""Caucasian""","""Non-Hispanic origin""","""KT00002""","""Flu-Year-1-Day-0""","""2019""",28,0
"""sound-life_flow-cytometry/PB1_…","""PB1_panel/young-adult_female_c…","""a8ac1cd5-57fd-41c5-885a-6641ff…","""PB1""","""B151""",375504,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1003""",1989,30,"""Caucasian""","""Non-Hispanic origin""","""KT00003""","""Flu-Year-1-Day-0""","""2019""",30,0
"""sound-life_flow-cytometry/PB1_…","""PB1_panel/young-adult_male_cmv…","""d2d84933-d635-46ae-92ca-e8a940…","""PB1""","""B151""",440624,"""Young Adult""","""Male""","""Negative""","""BR1""","""BR1004""",1989,30,"""Caucasian""","""Non-Hispanic origin""","""KT00004""","""Flu-Year-1-Day-0""","""2019""",30,0
"""sound-life_flow-cytometry/PB1_…","""PB1_panel/young-adult_female_c…","""69bead0b-082f-4ce6-82f2-f8c835…","""PB1""","""B151""",399536,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1005""",1992,27,"""Caucasian""","""Non-Hispanic origin""","""KT00006""","""Flu-Year-1-Day-0""","""2019""",27,0


In [66]:
file_df['out_dir'][0]

'sound-life_flow-cytometry/PB1_panel/young-adult_female_cmv-negative/'

Summarize subjects, samples, and events per group

In [67]:
file_df.group_by(
    ['subject.ageGroup', 'subject.biologicalSex', 'subject.cmv']
).agg(
    pl.col('subject.subjectGuid').unique().len().alias('n_subjects'),
    pl.col('sample.sampleKitGuid').len().alias('n_samples'),
    pl.col('flow.batchID').unique().len().alias('n_batches'),
    pl.col('flow.event_count').sum().alias('n_events'),
).sort(['subject.ageGroup', 'subject.biologicalSex', 'subject.cmv'])

subject.ageGroup,subject.biologicalSex,subject.cmv,n_subjects,n_samples,n_batches,n_events
str,str,str,u32,u32,u32,i64
"""Older Adult""","""Female""","""Negative""",10,91,45,32149280
"""Older Adult""","""Female""","""Positive""",17,163,66,58352730
"""Older Adult""","""Male""","""Negative""",12,116,52,40229136
"""Older Adult""","""Male""","""Positive""",8,80,40,29298416
"""Young Adult""","""Female""","""Negative""",18,162,51,61652496
"""Young Adult""","""Female""","""Positive""",10,76,35,30053680
"""Young Adult""","""Male""","""Negative""",12,107,45,40034690
"""Young Adult""","""Male""","""Positive""",9,73,29,29192829


In [68]:
file_df['flow.batchID'].unique().len()

103

### Build final output files and .tar bundles

Save the full set of metadata to both the archive staging folder, and as a separate file to store in HISE.

We'll drop the `out_dir` column before saving - this won't be very useful to end users, but we'll need it to write the group-specific metadata files. 

`file.name` will be relative to the final metadata file path.

In [69]:
fcs_meta_file = 'sound-life_flow-cytometry/PB1_fcs_sample_metadata_all.csv'
file_df.drop('out_dir').write_csv(fcs_meta_file)

out_fcs_meta = 'output/PB1_fcs_sample_metadata_all_{d}.csv'.format(d = date.today())
file_df.drop('out_dir').write_csv(out_fcs_meta)

In [70]:
out_tarfiles = []
for out_dir in file_df['out_dir'].unique():
    if out_dir is not None:
        out_base = os.path.basename(re.sub('/$','',out_dir))

        out_meta_file = f'sound-life_flow-cytometry/PB1_fcs_sample_metadata_{out_base}.csv'
        out_meta = file_df.filter(
            pl.col('out_dir') == out_dir
        ).drop('out_dir')
        
        out_meta.write_csv(out_meta_file)

        d = date.today()
        out_tarfile = f'output/sound-life_flow-cytometry_PB1_{out_base}_{d}.tar'
        out_tarfiles.append(out_tarfile)

        with tarfile.open(out_tarfile, 'w') as tar:
            tar.add(fcs_panel_file)
            tar.add(fcs_meta_file)
            tar.add(out_meta_file)
            for fcs_file in out_meta['file.name']:
                tar.add(f'sound-life_flow-cytometry/{fcs_file}')

## Upload .fcs data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [76]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life PB1 panel .fcs files {d}'.format(d = date.today())

In [71]:
search_id = element_id()
search_id

'manganese-europium-selenium'

In [72]:
in_files = [meta_uuid, panel_uuid] + pb1_desc['file.id'].to_list()
len(in_files)

870

In [73]:
out_files = [out_fcs_meta] + out_tarfiles
out_files

['output/PB1_fcs_sample_metadata_all_2026-03-14.csv',
 'output/sound-life_flow-cytometry_PB1_young-adult_male_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_older-adult_female_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_older-adult_male_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_older-adult_female_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_older-adult_male_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_young-adult_female_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_young-adult_female_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PB1_young-adult_male_cmv-positive_2026-03-14.tar']

In [74]:
import session_info
session_info.show()

In [77]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

2026-03-14 17:32:55,592 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling upload_files


Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


2026-03-14 17:33:25,242 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling get_default_store
2026-03-14 17:33:28,499 INFO [hisepy.logging:208] logging 4403 133741222831936 Finished get_default_store, success=True, time_elapsed=0.469s
2026-03-14 17:34:51,466 INFO [hisepy.logging:175] logging 4403 133741222831936 Calling conda_env_builds
2026-03-14 17:34:51,467 INFO [hisepy.logging:53] utils 4403 133741222831936 Starting conda environment build validation...
2026-03-14 17:34:51,932 INFO [hisepy.logging:75] utils 4403 133741222831936 Exporting conda environment from /home/workspace/environment/minimalv2...
2026-03-14 17:34:55,353 INFO [hisepy.logging:88] utils 4403 133741222831936 Removing hisepy references from exported environment file...
2026-03-14 17:34:55,362 INFO [hisepy.logging:99] utils 4403 133741222831936 Creating temporary conda environment at /tmp/conda_env_test_4azbrga4/env_e464ac1d48604f06b5d563f11d759d47...
2026-03-14 17:39:15,868 INFO [hisepy.logging:132] utils

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'cfaee1bf-d753-40a9-806a-ee0cb8a920f9',
 'ProcessId': 'ee018b43-c5c8-40f7-868e-4d53e911585e',
 'WorkflowId': '2340d7f3-b52f-4cb3-9511-a28c9324a10a',
 'FileIds': ['9bcd4783-fd13-4d89-8962-d611985b4d23',
  '3b83f7a7-74d6-42a0-badd-787035ebd9db',
  '31b8dd79-2dd6-4f94-a734-87d96b506861',
  'd3483d18-c3a8-4ba5-8eef-5717d4c81eda',
  '1df63b5d-411c-4a00-bbd4-1b8e4c4ee5fb',
  'aa4b13bf-09e0-4579-bc9f-a22350888322',
  '673d1a76-cb68-45d0-9128-bb87612c7540',
  '0bd73f2d-bd9a-498a-8daa-10640819b2c5',
  'dbdcd6c8-7e8f-4ca5-80f5-a4d80c02baae']}